# 03 · Análisis descriptivo — ENCC 2017

Este cuaderno describe con números las variables que usan las 10 preguntas de análisis (univariado y bivariado), compara grupos con y sin ponderar, y genera el informe automático para contrastarlo con lo que se encontró a mano.

Parte del archivo `datos/derivados/encc_2017_limpio.parquet`, que genera `02_limpieza_encc.ipynb`. Las decisiones de limpieza están en el registro de ese cuaderno.

## 1. Carga

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datos').is_dir())
enc = pd.read_parquet(RAIZ / 'datos' / 'derivados' / 'encc_2017_limpio.parquet')
assert len(enc) == 2800

NUM = ['edad', 'minutos_internet_total', 'minutos_redes_total', 'minutos_videojuegos_total']

## 2. Univariado

**Pregunta:** ¿cómo se distribuyen las variables numéricas de las preguntas P2, P4 y P8, y alcanza la media para describirlas?

In [6]:
def resumen(df, cols):
    filas = []
    for c in cols:
        s = df[c].dropna().astype(float)
        q1, q3 = s.quantile([.25, .75])
        filas.append({'variable': c, 'n': len(s), 'media': round(s.mean(), 1),
                      'mediana': s.median(), 'desvío': round(s.std(), 1), 'IQR': q3 - q1,
                      'mín': s.min(), 'máx': s.max(), '% ceros': round(100 * (s == 0).mean(), 1),
                      'asimetría': round(s.skew(), 2), 'curtosis': round(s.kurt(), 2)})
    return pd.DataFrame(filas).set_index('variable')

resumen(enc, NUM)

,n,media,mediana,desvío,IQR,mín,máx,% ceros,asimetría,curtosis
variable,,,,,,,,,,
edad,2800,39.1,36.0,19.3,33.0,13.0,92.0,0.0,0.47,-0.88
minutos_internet_total,2798,242.6,150.0,291.9,240.0,0.0,1440.0,19.3,2.05,4.61
minutos_redes_total,2798,140.2,60.0,233.2,180.0,0.0,1440.0,29.9,3.22,12.13
minutos_videojuegos_total,2798,26.2,0.0,68.0,0.0,0.0,1200.0,76.8,5.23,50.06


**Qué se lee:**

- En las tres variables de minutos la media está muy por encima de la mediana: hay muchos ceros ("no usa") y unos pocos valores enormes que tiran la media hacia arriba.
- En videojuegos, el 76,8 % tiene 0 y la mediana es 0. El promedio de 26 minutos no representa a una persona típica, así que conviene reportar también el porcentaje que juega y el promedio entre quienes juegan.
- `edad` es casi simétrica (asimetría 0,47), sin problemas.

In [7]:
for c in ['sexo', 'p10', 'p42', 'p63', 'p98', 'p23', 'NSEcat1', 'p106']:
    print(c, enc[c].value_counts(dropna=False).to_dict())

sexo {'Mujer': 1403, 'Varón': 1397}
p10 {'SI': 2581, 'NO': 219}
p42 {'NO': 1424, 'SI': 1359, None: 17}
p63 {'NO': 1692, 'SI': 1108}
p98 {'NO': 2371, 'SI': 429}
p23 {'NO': 2082, 'SI': 718}
NSEcat1 {'C3': 858, 'D1': 779, 'C2': 473, 'D2': 431, 'C1': 231, 'AB': 17, 'E': 9, nan: 2}
p106 {'SECUNDARIA INCOMPLETA': 783, 'SECUNDARIA COMPLETA': 700, 'PRIMARIA COMPLETA': 390, 'UNIVERSITARIO INCOMPLETO': 256, 'TERCIARIO COMPLETO': 199, 'UNIVERSITARIO COMPLETO': 180, 'TERCIARIO INCOMPLETO': 157, 'PRIMARIA INCOMPLETA': 94, 'SIN ESTUDIOS': 22, 'POSGRADO INCOMPLETO O COMPLETO': 19}


`NSEcat1` tiene categorías muy chicas (AB: 17 casos, E: 9) y `p106` también (sin estudios: 22, posgrado: 19). Si se cruzan con otra categórica, el chi² puede quedar con frecuencias esperadas menores a 5.

## 3. Bivariado

**Pregunta:** ¿qué variables se asocian entre sí, y qué pruebas se pueden usar con estos datos?

In [8]:
# Pearson y Spearman, porque las variables de minutos tienen muchos ceros y colas largas
display(enc[NUM].corr(method='pearson').round(2))
display(enc[NUM].corr(method='spearman').round(2))

,edad,minutos_internet_total,minutos_redes_total,minutos_videojuegos_total
edad,1.00,-0.35,-0.29,-0.31
minutos_internet_total,-0.35,1.00,0.60,0.19
minutos_redes_total,-0.29,0.60,1.00,0.22
minutos_videojuegos_total,-0.31,0.19,0.22,1.00


,edad,minutos_internet_total,minutos_redes_total,minutos_videojuegos_total
edad,1.00,-0.47,-0.50,-0.45
minutos_internet_total,-0.47,1.00,0.71,0.27
minutos_redes_total,-0.50,0.71,1.00,0.31
minutos_videojuegos_total,-0.45,0.27,0.31,1.00


In [9]:
def asociacion(df, a, b):
    t = pd.crosstab(df[a], df[b])
    chi2, p, gl, esp = stats.chi2_contingency(t)
    return {'cruce': f'{a} x {b}', 'chi2': round(float(chi2), 1), 'p': float(p), 'gl': int(gl),
            'celdas esperado<5': int((esp < 5).sum()), 'celdas': int(esp.size),
            'usable': bool((esp >= 5).all()),
            'V de Cramér': round(float(np.sqrt(chi2 / (t.values.sum() * (min(t.shape) - 1)))), 3)}

pd.DataFrame([asociacion(enc, a, b) for a, b in
              [('NSEcat1', 'region'), ('p63', 'region'), ('sexo', 'p42'), ('NSEcat1', 'p42')]]).set_index('cruce')

,chi2,p,gl,celdas esperado<5,celdas,usable,V de Cramér
cruce,,,,,,,
NSEcat1 x region,220.5,2.259647e-28,36,14,49,False,0.115
p63 x region,158.8,1.072658e-31,6,0,14,True,0.238
sexo x p42,26.0,3.395206e-07,1,0,4,True,0.097
NSEcat1 x p42,226.1,5.289685e-46,6,2,14,False,0.285


**Qué se lee:**

- Pearson y Spearman se separan mucho (por ejemplo, `edad` con `minutos_redes_total`: −0,29 contra −0,50). Con tantos ceros y colas largas, Pearson subestima la relación; conviene reportar los dos.
- `minutos_internet_total` y `minutos_redes_total` van juntas (0,60 y 0,71), lo esperable si las redes son parte del uso de internet.
- El chi² de `NSEcat1` con `region` **no se puede usar** (14 de 49 celdas con esperado < 5), y el de `NSEcat1` con `p42` tampoco (2 de 14). Hay que agrupar categorías.
- `sexo` con `p42` sí es usable. El p es muy chico pero V de Cramér vale 0,097: hay diferencia, pero es chica ("significativo, no importante").

## 4. Comparación de grupos, con y sin ponderar (registro #7)

**Preguntas:** P4 (¿la diferencia en minutos de redes entre mujeres y varones supera el 10 %?) y P6 (¿qué porcentaje fue al cine, CABA contra el resto?).

Cada región tiene 400 casos por diseño, así que la muestra no es proporcional a la población. `pondera_dem` corrige eso.

In [10]:
w = enc['pondera_dem'].astype(float)

def media_pond(x, peso):
    m = x.notna()
    return np.average(x[m], weights=peso[m])

def mediana_pond(x, peso):
    m = x.notna()
    x, peso = x[m].to_numpy(float), peso[m].to_numpy(float)
    orden = np.argsort(x)
    x, peso = x[orden], peso[orden]
    acum = np.cumsum(peso)
    return x[np.searchsorted(acum, acum[-1] / 2)]

# P4: minutos de redes por sexo
p4 = []
for s in ['Mujer', 'Varón']:
    d = enc[enc['sexo'] == s]
    p4.append({'sexo': s,
               'media sin ponderar': round(d['minutos_redes_total'].mean(), 1),
               'media ponderada': round(media_pond(d['minutos_redes_total'], w[d.index]), 1),
               'mediana sin ponderar': d['minutos_redes_total'].median(),
               'mediana ponderada': mediana_pond(d['minutos_redes_total'], w[d.index])})
p4 = pd.DataFrame(p4).set_index('sexo')
display(p4)
for col in ['media sin ponderar', 'media ponderada']:
    dif = 100 * (p4.loc['Mujer', col] - p4.loc['Varón', col]) / p4.loc['Varón', col]
    print(f'Diferencia relativa de la {col}: {dif:.1f} %')

# P6: % que fue al cine (p63), CABA contra el resto
p6 = []
for nombre, mascara in [('CABA', enc['region'] == 'CABA'), ('Resto', enc['region'] != 'CABA')]:
    d = enc[mascara]
    p6.append({'grupo': nombre,
               '% sin ponderar': round(100 * (d['p63'] == 'SI').mean(), 1),
               '% ponderado': round(100 * w[d.index][d['p63'] == 'SI'].sum() / w[d.index].sum(), 1)})
pd.DataFrame(p6).set_index('grupo')

,media sin ponderar,media ponderada,mediana sin ponderar,mediana ponderada
sexo,,,,
Mujer,154.1,141.3,60.0,60.0
Varón,126.1,95.8,60.0,30.0


Diferencia relativa de la media sin ponderar: 22.2 %
Diferencia relativa de la media ponderada: 47.5 %


,% sin ponderar,% ponderado
grupo,,
CABA,59.0,53.7
Resto,36.3,33.4


**Qué se lee:**

- **P4:** sin ponderar, las mujeres superan a los varones en un 22,2 % en la media, pero las medianas son iguales (60). Ponderando, la diferencia en la media sube a 47,5 % y las medianas se separan (60 contra 30). La respuesta a "¿supera el 10 %?" depende de si se usa media o mediana y de si se pondera, así que esa decisión tiene que quedar escrita.
- **P6:** CABA supera al resto con o sin ponderar (59,0 % contra 36,3 % sin ponderar; 53,7 % contra 33,4 % ponderado), pero el porcentaje baja en ambos grupos al ponderar.
- La mediana ponderada es una aproximación (primer valor donde el peso acumulado llega a la mitad).

## 5. Informe automático

El informe se escribe en `salidas/` como HTML y **no se versiona**: se regenera. Se abre en el navegador, no acá.

In [11]:
from data_profiling import ProfileReport

SALIDAS = RAIZ / 'salidas'
SALIDAS.mkdir(exist_ok=True)

CLAVE = ['edad', 'sexo', 'region', 'NSEcat1', 'p106', 'pondera_dem', 'p10', 'p42', 'p63', 'p98', 'p23',
         'minutos_internet_total', 'minutos_redes_total', 'minutos_videojuegos_total']

# Informe de las variables de las preguntas, completo (correlaciones e interacciones)
ProfileReport(enc[CLAVE].copy(), title='ENCC 2017 - variables clave', progress_bar=False).to_file(SALIDAS / 'perfil_encc_clave.html')

# Informe de las 443 columnas: tarda unos 5 minutos, por eso va apagado
COMPLETO = False
if COMPLETO:
    ProfileReport(enc.copy(), title='ENCC 2017 - completo', minimal=True, progress_bar=False).to_file(SALIDAS / 'perfil_encc_completo.html')

100%|██████████| 14/14 [00:00<00:00, 92.69it/s]


### El informe contra lo hallado a mano

*(Completar al abrir el informe.)*

**Alertas del informe que ya se habían detectado a mano:**

- (completar)

**Una alerta que no se había visto, y si resultó real:**

- (completar)

**Lo que se sabía y el informe no marca** (esta lista justifica el análisis exploratorio y va al informe del TP1):

- (completar)

## Decisiones nuevas para el registro

Se suman a la tabla del registro de decisiones (continúan la numeración de `02_limpieza_encc.ipynb`).

| # | Qué | Qué se hizo | Evidencia | Por qué (alternativa descartada) | Pregunta |
|---|---|---|---|---|---|
| 9 | `minutos_*_total` con muchos ceros (19–77 %) | Se reporta media y mediana, y en videojuegos también el % que juega | Media 26,2 y mediana 0 en videojuegos | La media sola engaña con colas largas | P2, P4, P8 |
| 10 | `NSEcat1` y `p106` en cruces con otras categóricas | (completar: qué categorías se agrupan) | 14 de 49 celdas con esperado < 5 en `NSEcat1` x `region` | El chi² no es válido con esperado < 5 | P3, P7 |
| 11 | P4 y P6: ponderar o no | (completar: qué se reporta) | Diferencia relativa en P4: 22,2 % sin ponderar, 47,5 % ponderada | Los pesos van de 2 a 182.969; sin ponderar, los porcentajes no son nacionales | P4, P6 |